# 2. Tính toán chỉ số đánh giá Chuẩn hóa nhãn kỹ năng (JD & CV) và gộp nhóm doanh nghiệp
Notebook này đối sánh kết quả thực tế với nhãn Ground Truth để tính toán `Mapping Accuracy`, `Reject Accuracy` cho cả JD, CV và `Company Match Accuracy` cho Doanh nghiệp.


## Bước 2.1: Nhập thư viện và nạp dữ liệu


In [2]:
import json
from pathlib import Path

workspace_root = Path(r'f:\HCMUS_KH\LuanVan\JobVisualization_BE')
script_dir = workspace_root / 'KiemThu' / 'KiemThu_SkillNormalization'

actual_file = script_dir / 'actual_normalization_results.json'
gt_file = script_dir / 'ground_truth_normalization.json'

with open(actual_file, 'r', encoding='utf-8') as f:
    actual = json.load(f)
with open(gt_file, 'r', encoding='utf-8') as f:
    gt = json.load(f)

print('✓ Đã nạp thành công dữ liệu thực tế và Ground Truth!')


✓ Đã nạp thành công dữ liệu thực tế và Ground Truth!


## Bước 2.2: Tính toán chỉ số đối soát Kỹ năng từ JD tuyển dụng


In [3]:
act_skills = actual['skills']
gt_skills = gt['skills']

act_skills_map = {(item['url'], item['skill_extract'].lower().strip()): item['skill_normalize'] for item in act_skills}

total_mapping_valid = 0
correct_mapping = 0
total_reject_valid = 0
correct_reject = 0
detailed_skill_table = []

for gt_item in gt_skills:
    url = gt_item['url']
    extract = gt_item['skill_extract']
    gt_norm = gt_item['skill_normalize']
    act_norm = act_skills_map.get((url, extract.lower().strip()), 'Not Found')
    
    is_junk = (gt_norm == 'Rác')
    is_new = (gt_norm == 'Skill mới')
    
    if is_junk or is_new:
        total_reject_valid += 1
        if act_norm is None:
            correct_reject += 1
            status = 'TP (Reject OK)'
        else:
            status = 'FP (Should Reject)'
    else:
        total_mapping_valid += 1
        if act_norm and act_norm.lower().strip() == gt_norm.lower().strip():
            correct_mapping += 1
            status = 'TP (Map OK)'
        else:
            status = 'FN (Map Fail)'
            
    detailed_skill_table.append({
        'extract': extract,
        'gt': gt_norm,
        'actual': act_norm,
        'status': status
    })

jd_mapping_accuracy = correct_mapping / total_mapping_valid if total_mapping_valid > 0 else 0
jd_reject_accuracy = correct_reject / total_reject_valid if total_reject_valid > 0 else 0

print('--- Kết quả chuẩn hóa kỹ năng JD ---')
print(f'Mapping Accuracy: {jd_mapping_accuracy*100:.2f}% ({correct_mapping}/{total_mapping_valid})')
print(f'Reject Accuracy : {jd_reject_accuracy*100:.2f}% ({correct_reject}/{total_reject_valid})')


--- Kết quả chuẩn hóa kỹ năng JD ---
Mapping Accuracy: 68.44% (167/244)
Reject Accuracy : 66.87% (224/335)


## Bước 2.3: Tính toán chỉ số đối soát Kỹ năng từ CV sinh viên


In [4]:
act_cv_skills = actual['cv_skills']
gt_cv_skills = gt['cv_skills']

act_cv_skills_map = {(item['filename'], item['skill_extract'].lower().strip()): item['skill_normalize'] for item in act_cv_skills}

total_cv_mapping_valid = 0
correct_cv_mapping = 0
total_cv_reject_valid = 0
correct_cv_reject = 0
detailed_cv_skill_table = []

for gt_item in gt_cv_skills:
    filename = gt_item['filename']
    extract = gt_item['skill_extract']
    gt_norm = gt_item['skill_normalize']
    act_norm = act_cv_skills_map.get((filename, extract.lower().strip()), 'Not Found')
    
    is_junk = (gt_norm == 'Rác')
    is_new = (gt_norm == 'Skill mới')
    
    if is_junk or is_new:
        total_cv_reject_valid += 1
        if act_norm is None:
            correct_cv_reject += 1
            status = 'TP (Reject OK)'
        else:
            status = 'FP (Should Reject)'
    else:
        total_cv_mapping_valid += 1
        if act_norm and act_norm.lower().strip() == gt_norm.lower().strip():
            correct_cv_mapping += 1
            status = 'TP (Map OK)'
        else:
            status = 'FN (Map Fail)'
            
    detailed_cv_skill_table.append({
        'extract': extract,
        'gt': gt_norm,
        'actual': act_norm,
        'status': status
    })

cv_mapping_accuracy = correct_cv_mapping / total_cv_mapping_valid if total_cv_mapping_valid > 0 else 0
cv_reject_accuracy = correct_cv_reject / total_cv_reject_valid if total_cv_reject_valid > 0 else 0

print('--- Kết quả chuẩn hóa kỹ năng CV ---')
print(f'Mapping Accuracy: {cv_mapping_accuracy*100:.2f}% ({correct_cv_mapping}/{total_cv_mapping_valid})')
print(f'Reject Accuracy : {cv_reject_accuracy*100:.2f}% ({correct_cv_reject}/{total_cv_reject_valid})')


--- Kết quả chuẩn hóa kỹ năng CV ---
Mapping Accuracy: 64.71% (44/68)
Reject Accuracy : 76.58% (85/111)


## Bước 2.4: Tính toán chỉ số khớp mờ Doanh nghiệp (Company Match Accuracy)


In [5]:
act_comps = actual['companies']
gt_comps = gt['companies']

act_comps_map = {(item['url'], item['company_raw'].lower().strip()): item['company_normalize'] for item in act_comps}

total_comps = 0
correct_comps = 0
detailed_comp_table = []

for gt_item in gt_comps:
    url = gt_item['url']
    raw = gt_item['company_raw']
    gt_norm = gt_item['company_normalize']
    act_norm = act_comps_map.get((url, raw.lower().strip()), 'Not Found')
    
    total_comps += 1
    if (gt_norm is None and act_norm is None) or (gt_norm and act_norm and gt_norm.lower().strip() == act_norm.lower().strip()):
        correct_comps += 1
        status = 'OK'
    else:
        status = 'Fail'
        
    detailed_comp_table.append({
        'raw': raw,
        'gt': gt_norm,
        'actual': act_norm,
        'status': status
    })

comp_accuracy = correct_comps / total_comps if total_comps > 0 else 0

print('--- Kết quả gộp nhóm doanh nghiệp ---')
print(f'Company Match Accuracy: {comp_accuracy*100:.2f}% ({correct_comps}/{total_comps})')


--- Kết quả gộp nhóm doanh nghiệp ---
Company Match Accuracy: 83.33% (20/24)


## Bước 2.5: Tổng hợp kết quả và xuất các bảng báo cáo luận văn


In [6]:
avg_mapping = (correct_mapping + correct_cv_mapping) / (total_mapping_valid + total_cv_mapping_valid)
avg_reject = (correct_reject + correct_cv_reject) / (total_reject_valid + total_cv_reject_valid)

print('='*80)
print('          TỔNG HỢP CÁC CHỈ SỐ ĐÁNH GIÁ CHUẨN HÓA CHUNG CUỘC')
print('='*80)
print(f'- Độ chính xác chuẩn hóa kỹ năng chung (Mapping Accuracy) : {avg_mapping*100:.2f}%')
print(f'- Độ chính xác lọc nhiễu kỹ năng chung (Reject Accuracy)   : {avg_reject*100:.2f}%')
print(f'- Độ chính xác gộp nhóm doanh nghiệp (Company Match Accuracy): {comp_accuracy*100:.2f}%')
print('='*80)

print('### Bảng đối soát chuẩn hóa mẫu (Kỹ năng từ JD):')
print('| STT | Kỹ năng thô (JD) | Nhãn chuẩn hóa kỳ vọng | Kết quả chuẩn hóa thực tế | Trạng thái |')
print('|---|---|---|---|---|')
for idx, row in enumerate(detailed_skill_table[:30], 1):
    print(f'| {idx} | {row["extract"]} | {row["gt"]} | {row["actual"]} | {row["status"]} |')

print('\n### Bảng đối soát chuẩn hóa mẫu (Kỹ năng từ CV):')
print('| STT | Kỹ năng thô (CV) | Nhãn chuẩn hóa kỳ vọng | Kết quả chuẩn hóa thực tế | Trạng thái |')
print('|---|---|---|---|---|')
for idx, row in enumerate(detailed_cv_skill_table[:30], 1):
    print(f'| {idx} | {row["extract"]} | {row["gt"]} | {row["actual"]} | {row["status"]} |')

print('\n### Bảng đối soát chuẩn hóa mẫu (Doanh nghiệp):')
print('| STT | Tên doanh nghiệp thô | Nhãn kỳ vọng | Nhãn thực tế | Trạng thái |')
print('|---|---|---|---|---|')
for idx, row in enumerate(detailed_comp_table, 1):
    print(f'| {idx} | {row["raw"]} | {row["gt"]} | {row["actual"]} | {row["status"]} |')


          TỔNG HỢP CÁC CHỈ SỐ ĐÁNH GIÁ CHUẨN HÓA CHUNG CUỘC
- Độ chính xác chuẩn hóa kỹ năng chung (Mapping Accuracy) : 67.63%
- Độ chính xác lọc nhiễu kỹ năng chung (Reject Accuracy)   : 69.28%
- Độ chính xác gộp nhóm doanh nghiệp (Company Match Accuracy): 83.33%
### Bảng đối soát chuẩn hóa mẫu (Kỹ năng từ JD):
| STT | Kỹ năng thô (JD) | Nhãn chuẩn hóa kỳ vọng | Kết quả chuẩn hóa thực tế | Trạng thái |
|---|---|---|---|---|
| 1 | Data Structures | Data Structures | Data Structures | TP (Map OK) |
| 2 | Algorithms | Algorithms | Algorithms | TP (Map OK) |
| 3 | Design Patterns | Skill mới | Software Design Patterns | FP (Should Reject) |
| 4 | Object-Oriented Programming Principles | Rác | Object Oriented Programming And Systems | FP (Should Reject) |
| 5 | Functional Programming Principles | Rác | Functional Programming | FP (Should Reject) |
| 6 | Programming Language Proficiency | Rác | Programming Language Design | FP (Should Reject) |
| 7 | Frameworks | Rác | UIKit (Apple App Fram